# Thesis Note 03.1 — Data Pipeline Architecture

## Milestone

Milestone 03 — Data Pipeline Design

Sub-Milestone 03.1 — Data Pipeline Architecture

---

# Objective

The objective of this milestone is to define the overall architecture of the data pipeline before any implementation begins. A well-designed data pipeline serves as the foundation for all subsequent stages of the project, including baseline models, self-supervised learning, multimodal fusion, and missing-modality experiments.

Instead of developing task-specific preprocessing code, the pipeline is designed to remain reusable, reproducible, and independent of any particular learning algorithm.

---

# Motivation

The COde dataset contains multiple levels of hierarchy:

* Patient
* Visit (Checkup)
* Images
* Clinical Text

In addition, previous audit studies revealed several important characteristics of the dataset:

* Multiple visits may belong to the same patient.
* Images can be reused across different visits of the same patient.
* Radiographs are naturally missing for a substantial proportion of visits.
* Clinical text is available for nearly every visit.
* A visit may contain multiple photographs and multiple radiographs.

These observations imply that the pipeline must preserve the hierarchical structure of the dataset while preventing information leakage between training and evaluation sets.

---

# Design Objectives

The proposed data pipeline should satisfy the following objectives:

* Fully reproducible
* Config-driven
* Modular
* Independent from downstream models
* Independent from downstream learning tasks
* Compatible with naturally missing modalities
* Compatible with patient-level data partitioning
* Easily extendable for future experiments

---

# Architecture Decisions

## AD-01 — Patient-Level Split

The patient is selected as the unit of dataset partitioning.

All visits belonging to a patient must remain in the same split (training, validation, or testing).

This decision prevents both patient-level information leakage and cross-visit duplicate image leakage.

Status:

Approved

---

## AD-02 — Visit as the Fundamental Sample

Although the split is performed at the patient level, the basic learning sample is defined at the visit level.

Each visit represents one multimodal observation of a patient at a particular time point.

Status:

Approved

---

## AD-03 — Multimodal Sample Representation

Each visit is represented as a single multimodal sample consisting of:

* Patient identifier
* Visit identifier
* Photograph paths
* Radiograph paths
* Clinical text
* Metadata
* Missing-modality indicators
* Raw labels

The pipeline therefore treats all available information belonging to a visit as one coherent sample.

Status:

Approved

---

## AD-04 — Model-Agnostic Design

The data pipeline must not contain any assumptions regarding the downstream learning model.

It should support conventional CNNs, Vision Transformers, multimodal encoders, self-supervised learning frameworks, and future architectures without modification.

Status:

Approved

---

## AD-05 — Task-Agnostic Design

The pipeline must remain independent of downstream prediction tasks.

Tasks such as diagnosis classification, multi-label prediction, retrieval, self-supervised learning, or missing-modality prediction should not influence the data loading process.

Task-specific processing will be implemented in a dedicated future milestone.

Status:

Approved

---

## AD-06 — Preserve Naturally Missing Modalities

Missing radiographs represent a genuine characteristic of the COde dataset rather than corrupted data.

Therefore:

* Missing samples must not be removed.
* Artificial imputation is not performed during data loading.
* Missing information is explicitly represented using modality flags.

This design enables future research on naturally incomplete multimodal learning.

Status:

Approved

---

## AD-07 — Deferred Label Processing

Diagnostic labels remain in their original form during the data pipeline stage.

Operations such as:

* label normalization
* synonym merging
* class filtering
* long-tail handling
* target definition

will be performed later during the Task Definition & Diagnostic Processing milestone.

Status:

Approved

---

## AD-08 — Single Responsibility Principle

Each component of the pipeline performs one clearly defined responsibility.

Examples include:

* CSV loading
* split assignment
* image loading
* clinical text loading
* metadata extraction
* missing-modality handling
* PyTorch dataset construction

This modular design simplifies maintenance and future extensions.

Status:

Approved

---

# Logical Data Hierarchy

The logical organization of the dataset is defined as follows:

Patient

↓

Visit

↓

Multimodal Sample

↓

Modalities

Each visit corresponds to exactly one multimodal sample.

The modalities associated with a visit may include:

* Clinical text
* Photographs
* Radiographs

depending on their availability.

---

# Data Flow

The proposed processing pipeline is summarized below.

COde Dataset

↓

Load CSV

↓

Load Patient-Level Split

↓

Assign Dataset Split

↓

Visit Records

↓

Build Multimodal Samples

↓

Image Loader

Clinical Text Loader

Metadata Loader

↓

Missing-Modality Handler

↓

PyTorch Dataset

↓

Transforms

↓

DataLoader

---

# Component Responsibilities

The pipeline is divided into independent components.

CSV Loader

Reads the raw dataset.

Split Loader

Loads patient-level split assignments.

Sample Builder

Constructs multimodal samples from visit records.

Image Loader

Loads image files and validates file availability.

Clinical Text Loader

Extracts clinical narratives.

Metadata Loader

Extracts auxiliary structured information.

Missing-Modality Handler

Computes explicit modality availability indicators.

Dataset

Provides samples for PyTorch.

Transforms

Applies image and text preprocessing.

DataLoader

Constructs mini-batches for model training.

---

# Expected Benefits

The proposed architecture provides several advantages.

* Prevents patient-level data leakage.
* Supports naturally missing modalities.
* Separates data processing from model implementation.
* Enables reproducible experimentation.
* Simplifies integration of multiple learning paradigms.
* Facilitates future extension without major refactoring.
* Provides a consistent data interface across all experiments.

---

# Relation to Subsequent Milestones

This architectural design serves as the foundation for the remaining implementation stages.

The following milestones will progressively implement the proposed design:

* Configuration System
* Multimodal Sample Representation
* Image Loading
* Clinical Text Processing
* Missing-Modality Handling
* PyTorch Dataset
* DataLoader Factory
* Pipeline Validation

No implementation is performed during this milestone.

The outcome of this stage is the finalized architectural specification that will guide all future development.


# Thesis Note 03.2 — Configuration System Design

## Milestone

Milestone 03 — Data Pipeline Design

Sub-Milestone 03.2 — Configuration System

---

# Objective

The objective of this milestone is to establish a lightweight and reproducible configuration system for the project pipeline.

The purpose of introducing configuration files is to separate implementation logic from experiment settings and environment-specific parameters.

The configuration system enables:

* reproducible experiments,
* easier hardware adaptation,
* reduced hard-coded parameters,
* simpler future extensions.

---

# Motivation

The project is designed to run initially on a local development environment with limited computational resources:

* NVIDIA RTX 3050 Laptop GPU
* 4GB VRAM
* WSL2 environment

However, future experiments may be executed on a more powerful server GPU.

Therefore, the pipeline requires a mechanism that allows changing execution environments without modifying the source code.

---

# Design Principle

The configuration system follows an incremental architecture strategy.

Instead of defining all possible future configurations at the beginning, only configurations required for the current development stage are introduced.

Future configuration files will be added when corresponding implementation components become necessary.

This prevents unnecessary complexity and avoids premature abstraction.

---

# Configuration Structure

The current configuration structure is:

```
configs/
│
├── audit.yaml
├── base.yaml
├── local_rtx3050.yaml
└── server.yaml
```

---

# Base Configuration

File:

```
configs/base.yaml
```

The base configuration contains shared project-level settings.

It defines:

* random seed,
* reproducibility settings,
* dataset paths,
* output directories,
* dataset hierarchy decisions,
* runtime defaults.

Current design:

```yaml
project:
  seed: 42
  deterministic: true

paths:
  dataset_csv: data/raw/COde-Dataset/complete_dataset.csv
  images_root: data/raw/COde-Dataset/Images
  patient_split: results/patient_level_split/patient_split.csv
  output_dir: results

dataset:
  sample_unit: visit
  split_unit: patient

runtime:
  device: auto
  num_workers: 4
  pin_memory: true
```

---

# Local RTX 3050 Configuration

File:

```
configs/local_rtx3050.yaml
```

The local configuration contains only values that differ from the base configuration.

This configuration targets development on:

* RTX 3050 Laptop GPU
* 4GB VRAM

Current overrides:

```yaml
runtime:
  device: cuda
  num_workers: 2
  pin_memory: true
```

The configuration intentionally avoids adding training-related parameters because model training has not started yet.

---

# Server Configuration

File:

```
configs/server.yaml
```

The server configuration is currently a placeholder.

No assumptions are made regarding:

* GPU model,
* VRAM capacity,
* CPU resources,
* storage system.

Only the expected CUDA execution environment is specified.

Future server-specific parameters will be added after the actual hardware configuration becomes available.

---

# Architecture Decisions

## AD-09 — Incremental Architecture & Configuration Design

Status:

Approved

Decision:

The project architecture and configuration system will evolve incrementally.

Only components required by the current milestone are implemented.

Rationale:

This project is research-oriented, and experimental requirements may change during development.

Prematurely designing all future modules increases unnecessary complexity and creates additional refactoring effort.

---

## AD-10 — Minimal Override Configuration

Status:

Approved

Decision:

Environment-specific configuration files should contain only parameters that differ from the base configuration.

Common values should remain in the base configuration.

Rationale:

This avoids duplicated configuration values and reduces inconsistency between different execution environments.

---

# Current Configuration Scope

At this stage, the configuration system covers:

* dataset paths,
* split paths,
* output paths,
* reproducibility settings,
* runtime environment.

It intentionally does not include:

* model parameters,
* optimizer settings,
* training parameters,
* SSL parameters,
* evaluation settings.

These will be introduced only when their corresponding implementation stages begin.

---

# Relation to Future Milestones

The configuration system will later be extended to support:

* dataset loading,
* image preprocessing,
* multimodal fusion,
* model selection,
* self-supervised learning,
* training experiments,
* evaluation protocols.

The current design provides the minimal foundation required for the next milestone:

Milestone 10.3 — Multimodal Sample Representation.

---

# Conclusion

A lightweight configuration framework was established to support reproducible and scalable development.

The current design separates environment-specific settings from implementation logic while maintaining flexibility for future research experiments.
